In [2]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([[1, 2], [3, 4], [5, 6]])
x.shape

y = np.array([10, 20])
assert y.shape == (2,)
y.ndim



1

In [3]:
def naive_relu(x):
 assert len(x.shape) == 2
 x = x.copy()
 for i in range(x.shape[0]):
  for j in range(x.shape[1]):
     x[i, j] = max(x[i, j], 0)
 return x

def naive_add(x, y):
 assert len(x.shape) == 2
 assert x.shape == y.shape
 x = x.copy()
 for i in range(x.shape[0]):
   for j in range(x.shape[1]):
     x[i, j] += y[i, j]
 return x


In [4]:
import time
x = np.random.random((20, 100))
y = np.random.random((20, 100))

print(x.shape, y.shape)
print(x.ndim, y.ndim)

t0 = time.time()
for _ in range(1000):
 z = x + y
 z = np.maximum(z, 0.)
print("Took: {0:.6f} s".format(time.time() - t0))

(20, 100) (20, 100)
2 2
Took: 0.001986 s


In [11]:
x = np.random.random((20, 100))
y = np.random.random((20, 100))

print(len(x.shape))

t0 = time.time()
for _ in range(1000):
 z = naive_add(x, y)
 z = naive_relu(z)
print("Took: {0:.6f} s".format(time.time() - t0))

2
Took: 0.560961 s


In [14]:
import numpy as np
x = np.random.random((64, 3, 32, 10))
y = np.random.random((32, 10))
z = np.maximum(x, y)

print(z)

[[[[0.85391257 0.98592286 0.85380783 ... 0.72029542 0.44993086
    0.46962456]
   [0.81829671 0.83116443 0.89239447 ... 0.89256554 0.6736525
    0.30220655]
   [0.62779241 0.50263146 0.21845185 ... 0.95221041 0.51696654
    0.88648393]
   ...
   [0.62493753 0.527026   0.28737767 ... 0.89354154 0.76262053
    0.87366853]
   [0.99454983 0.72746838 0.77677982 ... 0.4662801  0.91245691
    0.69695631]
   [0.43339481 0.96920337 0.20520133 ... 0.94524545 0.48005403
    0.20026934]]

  [[0.2188588  0.98209311 0.69992334 ... 0.71206861 0.63634506
    0.46962456]
   [0.12042691 0.52461414 0.73365366 ... 0.64723487 0.6736525
    0.47898371]
   [0.62779241 0.50263146 0.7375262  ... 0.87445711 0.29311154
    0.4273109 ]
   ...
   [0.5013291  0.12286719 0.16433117 ... 0.8749964  0.99176135
    0.87366853]
   [0.3804044  0.43930261 0.77677982 ... 0.42763989 0.6259737
    0.57657188]
   [0.43339481 0.81128076 0.54628603 ... 0.45818244 0.48005403
    0.18855766]]

  [[0.07442325 0.98887003 0.49016042 

In [3]:
import os
# Wymuś CPU - GPU (compute capability 12.0) nie jest jeszcze w pełni wspierane przez TF
# MUSI być przed importem tensorflow!
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf

x = tf.Variable(tf.random.uniform((2, 2)))
print(x)
with tf.GradientTape() as tape:
 y = 2 * x + 3
print(y)
grad_of_y_wrt_x = tape.gradient(y, x)
print("Gradient:", grad_of_y_wrt_x)


<tf.Variable 'Variable:0' shape=(2, 2) dtype=float32, numpy=
array([[0.81852746, 0.2671727 ],
       [0.12750626, 0.16689038]], dtype=float32)>
tf.Tensor(
[[4.637055  3.5343454]
 [3.2550125 3.3337808]], shape=(2, 2), dtype=float32)
Gradient: tf.Tensor(
[[2. 2.]
 [2. 2.]], shape=(2, 2), dtype=float32)


In [16]:
import tensorflow as tf
import numpy as np

class NaiveDense:
 def __init__(self, input_size, output_size, activation):
    self.activation = activation
    w_shape = (input_size, output_size)
    w_initial_value = tf.random.uniform(w_shape, minval=0, maxval=1e-1)
    self.W = tf.Variable(w_initial_value)
    b_shape = output_size
    b_initial_value = tf.zeros(b_shape)
    self.b = tf.Variable(b_initial_value)

 def __call__(self, inputs):
    return self.activation(tf.matmul(inputs, self.W) + self.b)

 @property
 def weights(self):
   return [self.W, self.b]


class NaiveSequential:
    def __init__(self, layers):
        self.layers = layers
    def __call__(self, inputs):
        x = inputs
        for layer in self.layers:
            x = layer(x)
        return x

    @property
    def weights(self):
        weights = []
        for layer in self.layers:
            weights += layer.weights
        return weights


model = NaiveSequential([
    NaiveDense(input_size=28 * 28, output_size=512, activation=tf.nn.relu),
    NaiveDense(input_size=512, output_size=10, activation=tf.nn.softmax)
 ])

assert len(model.weights) == 4

import math
class BatchGenerator:
 def __init__(self, images, labels, batch_size=128):
    assert len(images) == len(labels)
    self.index = 0
    self.images = images
    self.labels = labels
    self.batch_size = batch_size
    self.num_batches = math.ceil(len(images) / batch_size)

 def next(self):
    images = self.images[self.index : self.index + self.batch_size]
    labels = self.labels[self.index : self.index + self.batch_size]
    self.index += self.batch_size
    return images, labels

def one_training_step(model, images_batch, labels_batch):
    with tf.GradientTape() as tape:
        predictions = model(images_batch)
        per_sample_losses = tf.keras.losses.sparse_categorical_crossentropy(
                                labels_batch, predictions)
        average_loss = tf.reduce_mean(per_sample_losses)
        gradients = tape.gradient(average_loss, model.weights)
        update_weights(gradients, model.weights)
    return average_loss


learning_rate = 1e-3
# def update_weights(gradients, weights):
#     for g, w in zip(gradients, weights):
#         w.assign_sub(g * learning_rate)

from tensorflow.keras import optimizers
optimizer = optimizers.SGD(learning_rate=learning_rate)
def update_weights(gradients, weights):
    optimizer.apply_gradients(zip(gradients, weights))


def fit(model, images, labels, epochs, batch_size=128):
 for epoch_counter in range(epochs):
    print(f"Epoch {epoch_counter}")
    batch_generator = BatchGenerator(images, labels)
    for batch_counter in range(batch_generator.num_batches):
        images_batch, labels_batch = batch_generator.next()
        loss = one_training_step(model, images_batch, labels_batch)
        if batch_counter % 100 == 0:
            print(f"loss at batch {batch_counter}: {loss:.2f}")

from tensorflow.keras.datasets import mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255
fit(model, train_images, train_labels, epochs=20, batch_size=128)

predictions = model(test_images)
predictions = predictions.numpy()
predicted_labels = np.argmax(predictions, axis=1)
matches = predicted_labels == test_labels
print(f"accuracy: {matches.mean():.2f}")




Epoch 0
loss at batch 0: 4.98
loss at batch 100: 2.23
loss at batch 200: 2.21
loss at batch 300: 2.10
loss at batch 400: 2.25
Epoch 1
loss at batch 0: 1.91
loss at batch 100: 1.87
loss at batch 200: 1.82
loss at batch 300: 1.72
loss at batch 400: 1.85
Epoch 2
loss at batch 0: 1.58
loss at batch 100: 1.57
loss at batch 200: 1.50
loss at batch 300: 1.42
loss at batch 400: 1.53
Epoch 3
loss at batch 0: 1.32
loss at batch 100: 1.34
loss at batch 200: 1.24
loss at batch 300: 1.20
loss at batch 400: 1.29
Epoch 4
loss at batch 0: 1.13
loss at batch 100: 1.16
loss at batch 200: 1.05
loss at batch 300: 1.04
loss at batch 400: 1.12
Epoch 5
loss at batch 0: 0.98
loss at batch 100: 1.02
loss at batch 200: 0.91
loss at batch 300: 0.92
loss at batch 400: 1.00
Epoch 6
loss at batch 0: 0.88
loss at batch 100: 0.91
loss at batch 200: 0.81
loss at batch 300: 0.83
loss at batch 400: 0.91
Epoch 7
loss at batch 0: 0.80
loss at batch 100: 0.83
loss at batch 200: 0.73
loss at batch 300: 0.76
loss at batch 40

In [56]:
tV=tf.Variable(initial_value=tf.random.normal(shape=(2, 3)))
print(tV)

tV.assign(tf.ones(shape=(2, 3)))
print(tV)

tV.assign_add(tf.ones(shape=(2, 3)))

tV[1,].assign(42)
print(tV)




<tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=
array([[-0.13523425,  0.2884324 ,  0.3866983 ],
       [ 0.18854123, -0.35786912, -0.24808504]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=
array([[1., 1., 1.],
       [1., 1., 1.]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=
array([[ 2.,  2.,  2.],
       [42., 42., 42.]], dtype=float32)>
